# Test the code from file `optimization.py`

We test the behavour of the optimization function `opt_moment_preserving_ent`.

- `opt_moment_preserving_ent`: Find the density matrix with the target Bloch lengths that minimizes (maximizes) a given entanglement measure. It takes several inputs.
    - dim : list[int]. List of local dimensions of the quantum system.
    - tensor_basis : np.ndarray. Tensor-product operator basis, shape (k, dn, dn).
    - subset_index_map : Dict[Tuple[int, ...], np.ndarray]. Mapping from subsystem subsets to indices in tensor_basis.
    - Rt : Dict[Tuple[int, ...], float]. Target Bloch vector norms for each subsystem subset.
    - metric : str, default="negativity". The entanglement metric to optimize.
    - optimization : str, default="minimize". Whether to 'minimize' or 'maximize' the metric.
    - cholesky_opt : bool, default=True. If True, use Cholesky parametrization for optimization.
    - exact_jac : bool, default=False. If True, compute the exact Jacobian for the trace norm metric.
    - purity_tol : float, default=1e-10. Tolerance for purity checks.
    - psd_tol : float, default=1e-10. Tolerance for positive semidefinite checks.
    - jac_tol : float, default=1e-10. Tolerance for Jacobian calculations.
    - local_maxiter : int, default=500. Maximum iterations for the local optimizer.

We are checking if the algorithm actually is optimizing. To do so, we choose a single point, perform the optimization and check the results.

# Importations

In [ ]:
# Numerical and scientific python programming
import numpy as np

# Local importations
from moments.bloch import generate_pauli_basis, compute_tensor_basis, compute_subset_index_map
from moments.optimization import opt_moment_preserving_ent

# Optimization

In [ ]:
# Define system parameters.
dim = [2, 2]
N = len(dim)

# Initialize the Pauli basis of a one-qubit system.
pauli_basis = generate_pauli_basis()
local_bases = [pauli_basis.copy()] * N
local_basis_sizes = [len(basis) for basis in local_bases]


# Compute tensor basis of the N qubit system.
tensor_basis = compute_tensor_basis(local_bases)
# Compute index mappings from basis elements to Bloch vector elements.
subset_index_map = compute_subset_index_map(local_basis_sizes)

# Define target Bloch lengts
Rt = {(1,): 0.5, (2,): 0.5, (1, 2): 1}

# Define keyword arguments for the optimizer
run_kwargs = {"dim": dim, "tensor_basis": tensor_basis, "subset_index_map": subset_index_map, "Rt": Rt,
              "optimization": "minimize", "metric": "partial_trace_norm", "cholesky_opt": True, "exact_jac": True}

In [ ]:
# Run the optimization
optimization_result = opt_moment_preserving_ent(**run_kwargs)

# Validation

In [ ]:
# Extract check data
checks = optimization_result.checks

# Compute validation metrics
bloch_equal = []
bloch_diff = {}
moments_equal = []
for subset in optimization_result.bloch_initial.keys():
        bloch_equal.append(np.allclose(optimization_result.bloch_initial[subset], optimization_result.bloch_final[subset]))
        bloch_diff[subset] = float(np.linalg.norm(optimization_result.bloch_initial[subset] - optimization_result.bloch_final[subset]))
        moments_equal.append(np.allclose(optimization_result.moments_initial[subset], optimization_result.moments_final[subset]))

print("Is the output a valid density matrix?", checks["is_valid_dm"])

print("\nAre the density matrices equal?", np.allclose(optimization_result.rho_initial, optimization_result.rho_final))
print("Density matrix difference:", np.linalg.norm(optimization_result.rho_initial - optimization_result.rho_final))

print("\nAre the Bloch vectors equal?", all(bloch_equal))
print("Bloch vector difference:", bloch_diff)

print("\nAre the Bloch lengths equal?", checks["moments_equal"])
print("Bloch lengths difference?", checks["moments_distance"])


print("\nIs the metric equal?", np.isclose(optimization_result.metric_initial, optimization_result.metric_final))
print("Metric difference:", abs(optimization_result.metric_initial - optimization_result.metric_final))

In [ ]:
# Display optimizer information
optimizer_info = optimization_result.optimizer_info
print("Result success:", optimizer_info["result_success"])
print("Result message:", optimizer_info["result_message"])

In [ ]:
# Display initial and final metrics
print(optimization_result.metric_initial)
print(optimization_result.metric_final)